# 01 · Modelo en estrella para Power BI y Tableau

Construye las cuatro tablas de `data/modelo/` a partir de ficheros versionados:

- `dim_equipo.csv`, `dim_posicion.csv`, `dim_jugador.csv`
- `hechos_jugador.csv`: métricas por 90 minutos y percentiles por grupo de posición

El cruce de nombres entre fuentes no se hace aquí: se lee de `data/mapa_nombres.csv`, que genera y
documenta el cuaderno 00.

**Cobertura:** jugadores de campo con minutos en LaLiga 2025/26. Los porteros quedan fuera de esta versión.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas numpy unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos numpy para los percentiles
import numpy as np
# Importamos pandas para trabajar con tablas
import pandas as pd
# Importamos unidecode para generar nombres de fichero sin acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)

# Minutos mínimos para entrar en el cálculo de percentiles (igual que en el TFM)
MINUTOS_MINIMOS = 600
# Grupos que nunca entran en el cálculo de percentiles
GRUPOS_SIN_PERCENTIL = ["sin_definir", "sin_confirmar"]

## 1. Carga

In [3]:
# Mapa de nombres: un jugador del modelo por fila
mapa = pd.read_csv("data/mapa_nombres.csv", dtype={"id_whoscored": "Int64"})
# Acciones de Opta por identificador de WhoScored
eventos = pd.read_csv("data/eventos_por_jugador.csv", dtype={"id_whoscored": "Int64"})
# Posiciones deducidas de las alineaciones
pos_ws = pd.read_csv("data/posiciones_alineaciones.csv")
# Jugadores de mi fichero de posiciones que no jugaron
sin_minutos = pd.read_csv("data/sin_minutos.csv")
# Cruces dudosos que no se aplican
descartados = pd.read_csv("data/cruces_descartados.csv")
# Fichero propio de posiciones
posiciones = pd.read_csv("data/origen/posiciones_laliga_2526.csv")
# Jugadores de Understat
understat = pd.read_csv("data/origen/jugadores_laliga_2526.csv")
# Altura media del bloque defensivo de cada equipo
altura = pd.read_csv("data/origen/altura_defensiva_laliga_2526.csv")
# Clasificación real y por puntos esperados
clasificacion = pd.read_csv("data/origen/clasificacion_xpts_laliga_2526.csv")

## 2. Cuadre de jugadores

In [4]:
# Descartados del cruce posiciones-Understat (salen del fichero de posiciones)
desc_pos = descartados[descartados["cruce"] == "posiciones-understat"]
# Jugadores del fichero de posiciones que entran en el modelo
entran_pos = mapa["nombre_posiciones"].notna().sum()
# Jugadores de campo recuperados de Understat sin posición
recuperados_us = mapa["nombre_posiciones"].isna().sum()
# Filas de Understat que no están en el modelo
fuera_us = understat[~understat["player_id"].isin(mapa["id_understat"])]

# Imprimimos el cuadre línea a línea
print(f"  jugadores en el fichero de posiciones          {len(posiciones):>4}")
print(f"- no jugaron (data/sin_minutos.csv)              {len(sin_minutos):>4}")
print(f"- descartados por cruce dudoso                   {len(desc_pos):>4}")
print(f"= jugadores del fichero de posiciones que entran {len(posiciones) - len(sin_minutos) - len(desc_pos):>4}")
print(f"+ jugadores de campo recuperados de Understat    {recuperados_us:>4}")
print(f"= total de jugadores del modelo                  {len(posiciones) - len(sin_minutos) - len(desc_pos) + recuperados_us:>4}")
print()
print(f"  (aparte) porteros excluidos                    {len(fuera_us):>4}")

# El cuadre tiene que coincidir con las filas del mapa
assert len(posiciones) - len(sin_minutos) - len(desc_pos) == entran_pos, "no cuadra el fichero de posiciones"
# Todo lo que queda fuera de Understat tienen que ser porteros
assert fuera_us["position"].str.contains("GK").all(), "hay jugadores de campo de Understat fuera del modelo"
# Comprobación desde el lado de Understat: cada fila está en el modelo o es un portero excluido
assert len(mapa) + len(fuera_us) == len(understat), "no cuadra Understat"
print(f"\n  comprobación Understat: {len(mapa)} en el modelo + {len(fuera_us)} porteros = {len(understat)} filas")

# Desglose del modelo por origen del cruce
print("\nDesglose por origen del cruce")
print(mapa["origen"].value_counts().to_string())

# Posiciones deducidas de las alineaciones
print("\nPosición deducida de las alineaciones:", len(pos_ws),
      "· confirmada:", int(pos_ws["confirmado"].sum()), "· sin confirmar:", int((~pos_ws["confirmado"]).sum()))

  jugadores en el fichero de posiciones           637
- no jugaron (data/sin_minutos.csv)               112
- descartados por cruce dudoso                      2
= jugadores del fichero de posiciones que entran  523
+ jugadores de campo recuperados de Understat      38
= total de jugadores del modelo                   561

  (aparte) porteros excluidos                      39

  comprobación Understat: 561 en el modelo + 39 porteros = 600 filas

Desglose por origen del cruce
origen
directo                   466
recuperado_36              36
understat_sin_posicion     36
recuperado_15              15
traspasado                  6
recuperado_grafia           2

Posición deducida de las alineaciones: 38 · confirmada: 10 · sin confirmar: 28


## 3. dim_equipo

Correspondencia explícita de nombres de equipo. Los CSV de origen no se tocan.

In [5]:
# Nombres de la altura defensiva y de WhoScored que no coinciden con mi fichero
EQUIPOS = {
    # Abreviatura de WhoScored
    "Atletico": "Atletico Madrid",
    # Nombre oficial completo del Alavés
    "Deportivo Alaves": "Alaves",
}
# Aplicamos la correspondencia a la altura defensiva
altura["equipo"] = altura["equipo"].replace(EQUIPOS)
# Aplicamos la correspondencia a la clasificación
clasificacion["equipo"] = clasificacion["equipo"].replace(EQUIPOS)

# Los 20 equipos, ordenados alfabéticamente
equipos = sorted(posiciones["equipo"].unique())
# Comprobamos que las tres fuentes de equipo coinciden
assert set(altura["equipo"]) == set(equipos) == set(clasificacion["equipo"]), "nombres de equipo distintos"

# Creamos la dimensión con una clave numérica
dim_equipo = pd.DataFrame({"id_equipo": range(1, len(equipos) + 1), "equipo": equipos})
# Añadimos la altura defensiva
dim_equipo = dim_equipo.merge(altura, on="equipo", how="left")
# Añadimos los puntos reales y los esperados
dim_equipo = dim_equipo.merge(clasificacion[["equipo", "puntos", "xpts"]].rename(
    columns={"xpts": "puntos_esperados"}), on="equipo", how="left")
# Nombre de fichero del escudo: minúsculas, sin acentos y con guiones bajos
dim_equipo["escudo"] = [unidecode(e).lower().replace(" ", "_") + ".png" for e in dim_equipo["equipo"]]
# Color primario, extraído del escudo en el cuaderno 03
escudos = pd.read_csv("data/escudos.csv")
# Lo añadimos por nombre de equipo
dim_equipo = dim_equipo.merge(escudos[["equipo", "color_primario"]], on="equipo", how="left")
# Los 20 equipos tienen que tener color
assert dim_equipo["color_primario"].notna().all(), "falta el color de algún equipo"
# Mostramos la tabla
print(dim_equipo.to_string(index=False))

 id_equipo          equipo  altura_defensiva  puntos  puntos_esperados              escudo color_primario
         1          Alaves             32.40      43             52.58          alaves.png        #0033a0
         2   Athletic Club             37.73      45             56.90   athletic_club.png        #e11f1c
         3 Atletico Madrid             33.29      69             61.83 atletico_madrid.png        #282a6f
         4       Barcelona             36.49      94             79.06       barcelona.png        #deb500
         5      Celta Vigo             31.25      54             51.37      celta_vigo.png        #ff0000
         6           Elche             32.00      43             41.67           elche.png        #e0cf9c
         7        Espanyol             33.24      46             44.13        espanyol.png        #e01517
         8          Getafe             32.85      51             42.95          getafe.png        #0062a5
         9          Girona             30.36  

## 4. dim_posicion

In [6]:
# Cada posición y su grupo de comparación; pivote y mediocentro van juntos, como en el TFM
POSICIONES = [
    ("central", "central"),
    ("lateral", "lateral"),
    ("pivote", "pivote_mediocentro"),
    ("mediocentro", "pivote_mediocentro"),
    ("mediapunta", "mediapunta"),
    ("extremo", "extremo"),
    ("delantero", "delantero"),
    ("sin_definir", "sin_definir"),
    ("sin_confirmar", "sin_confirmar"),
]
# Creamos la dimensión con una clave numérica
dim_posicion = pd.DataFrame(POSICIONES, columns=["posicion", "grupo"])
# Insertamos la clave al principio
dim_posicion.insert(0, "id_posicion", range(1, len(dim_posicion) + 1))
# Mostramos la tabla
print(dim_posicion.to_string(index=False))

 id_posicion      posicion              grupo
           1       central            central
           2       lateral            lateral
           3        pivote pivote_mediocentro
           4   mediocentro pivote_mediocentro
           5    mediapunta         mediapunta
           6       extremo            extremo
           7     delantero          delantero
           8   sin_definir        sin_definir
           9 sin_confirmar      sin_confirmar


## 5. dim_jugador

In [7]:
# Partimos del mapa
jug = mapa.copy()
# Nombre del jugador en el modelo: el de mi fichero si existe, si no el de Understat
jug["jugador"] = jug["nombre_posiciones"].fillna(jug["nombre_understat"])

# Posición del fichero propio para quien está en él
jug = jug.merge(posiciones.rename(columns={"jugador": "nombre_posiciones"}),
                on=["nombre_posiciones", "equipo"], how="left")
# Origen de la posición para los del fichero propio
jug["posicion_origen"] = np.where(jug["nombre_posiciones"].notna(), "fichero_propio", "alineaciones_whoscored")

# Posiciones de las alineaciones confirmadas por David
confirmadas = pos_ws[pos_ws["confirmado"]].set_index("id_understat")["posicion_propuesta"]
# Los recuperados de Understat toman su posición confirmada o quedan sin confirmar
sin_pos = jug["posicion_origen"] == "alineaciones_whoscored"
# Asignamos la posición
jug.loc[sin_pos, "posicion"] = jug.loc[sin_pos, "id_understat"].map(confirmadas).fillna("sin_confirmar")

# Reparto de partidos con eventos por club (útil sobre todo en los traspasados)
jug = jug.merge(eventos[["id_whoscored", "equipos_eventos"]], on="id_whoscored", how="left")
# Lo llamamos como se acordó
jug = jug.rename(columns={"equipos_eventos": "equipos_temporada", "minutos_us": "minutos", "partidos_us": "partidos"})

# Clave propia: ordenamos por equipo, nombre e identificador de Understat para que sea estable
jug = jug.sort_values(["equipo", "jugador", "id_understat"]).reset_index(drop=True)
# Numeramos desde 1
jug["id_jugador"] = range(1, len(jug) + 1)
# Traemos la clave de equipo
jug = jug.merge(dim_equipo[["id_equipo", "equipo"]], on="equipo", how="left")
# Traemos la clave de posición y su grupo
jug = jug.merge(dim_posicion, on="posicion", how="left")

# Columnas de la dimensión
COLUMNAS_DIM = ["id_jugador", "jugador", "id_equipo", "id_posicion", "minutos", "partidos",
                "posicion_origen", "equipos_temporada", "id_understat", "id_whoscored"]
# Construimos la dimensión
dim_jugador = jug[COLUMNAS_DIM].copy()
# Mostramos un vistazo
print(dim_jugador.head().to_string(index=False))

 id_jugador        jugador  id_equipo  id_posicion  minutos  partidos posicion_origen equipos_temporada  id_understat  id_whoscored
          1   Abde Rebbach          1            6     1479        30  fichero_propio         Alaves 30         11825        447835
          2    Aitor Mañas          1            7      105         9  fichero_propio          Alaves 9         14222        514695
          3  Ander Guevara          1            4      581        25  fichero_propio         Alaves 25          6126        344120
          4 Antonio Blanco          1            3     3137        36  fichero_propio         Alaves 36          9280        409467
          5         Calebe          1            5      767        19  fichero_propio         Alaves 19         13764        397568


## 6. hechos_jugador: métricas por 90 minutos

- De Understat: goles, xG, asistencias, xA, tiros, pases clave, xGChain y xGBuildup (y sus versiones sin penaltis).
- De Opta, por identificador de WhoScored: entradas intentadas y exitosas, intercepciones, despejes, pases
  bloqueados, duelos aéreos disputados y ganados, centros y toques en el área.
- Dos porcentajes de éxito, que no van por 90: `pct_entradas` y `pct_duelos_aereos`.

Si un jugador no tiene identificador de WhoScored, sus métricas de Opta quedan **vacías**, nunca a cero:
`tiene_eventos` distingue "no hizo la acción" (0) de "no tengo el dato" (vacío).

In [8]:
# Métricas de Understat y su nombre en el modelo
UNDERSTAT = {"goals": "goles", "np_goals": "goles_sin_penalti", "xg": "xg", "np_xg": "xg_sin_penalti",
             "assists": "asistencias", "xa": "xa", "shots": "tiros", "key_passes": "pases_clave",
             "xg_chain": "xg_chain", "xg_buildup": "xg_buildup"}
# Métricas de Opta (ya con su nombre en el modelo)
OPTA = ["entradas_intentadas", "entradas_exitosas", "intercepciones", "despejes", "pases_bloqueados",
        "duelos_aereos_disputados", "duelos_aereos_ganados", "centros", "toques_area"]
# Porcentajes de éxito: (columna, aciertos, intentos)
PORCENTAJES = [("pct_entradas", "entradas_exitosas", "entradas_intentadas"),
               ("pct_duelos_aereos", "duelos_aereos_ganados", "duelos_aereos_disputados")]

# Totales de Understat de cada jugador
hechos = jug[["id_jugador", "id_understat", "id_whoscored", "minutos", "grupo"]].merge(
    understat[["player_id"] + list(UNDERSTAT)].rename(columns={"player_id": "id_understat", **UNDERSTAT}),
    on="id_understat", how="left", validate="one_to_one")
# Totales de Opta; quien no tiene identificador se queda sin fila y sus valores quedan vacíos
hechos = hechos.merge(eventos[["id_whoscored"] + OPTA], on="id_whoscored", how="left")
# Tiene eventos quien tiene identificador de WhoScored
hechos["tiene_eventos"] = hechos["id_whoscored"].notna()

# Métricas que se pasan a valores por 90 minutos
POR90 = list(UNDERSTAT.values()) + OPTA
# Calculamos cada métrica por 90 minutos; un vacío sigue vacío
for m in POR90:
    # Total dividido entre minutos y multiplicado por 90
    hechos[m + "_90"] = hechos[m] / hechos["minutos"] * 90

# Calculamos los porcentajes de éxito; sin intentos, el porcentaje queda vacío (no es un 0 %)
for col, aciertos, intentos in PORCENTAJES:
    # Aciertos entre intentos, en tanto por cien
    hechos[col] = hechos[aciertos] / hechos[intentos].where(hechos[intentos] > 0) * 100

# Columnas de métrica del modelo: por 90 y porcentajes
COLUMNAS_METRICA = [m + "_90" for m in POR90] + [c for c, _, _ in PORCENTAJES]
# Columnas que salen de Opta
COLUMNAS_OPTA = [m + "_90" for m in OPTA] + [c for c, _, _ in PORCENTAJES]

# Control: los jugadores sin eventos tienen las métricas de Opta vacías
assert hechos.loc[~hechos["tiene_eventos"], COLUMNAS_OPTA].isna().all().all()
# Los que tienen eventos no tienen huecos en las métricas por 90
assert hechos.loc[hechos["tiene_eventos"], [m + "_90" for m in OPTA]].notna().all().all()
# Jugadores con eventos pero sin intentos, cuyo porcentaje queda vacío
for col, _, intentos in PORCENTAJES:
    # Recuento por porcentaje
    print(f"{col}: {int((hechos['tiene_eventos'] & hechos[col].isna()).sum())} jugadores con eventos y 0 {intentos}")

pct_entradas: 35 jugadores con eventos y 0 entradas_intentadas


pct_duelos_aereos: 39 jugadores con eventos y 0 duelos_aereos_disputados


## 7. Percentiles por grupo de posición

- **Grupo de referencia:** jugadores de su mismo grupo de posición con 600 minutos o más.
- **Fórmula:** porcentaje de jugadores del grupo de referencia con un valor **estrictamente menor**. Es la
  fórmula de las cifras publicadas en el TFM (Valjent p9, Llorente p98). El mejor del grupo no llega a 100.
- La misma fórmula se aplica a todos los jugadores del grupo, también a los de menos de 600 minutos,
  que se comparan contra el grupo de referencia pero llevan `percentil_fiable` en falso.
- `sin_definir` y `sin_confirmar` no tienen percentil.

In [9]:
# Un jugador es de referencia si tiene 600 minutos y un grupo con percentil
hechos["referencia"] = (hechos["minutos"] >= MINUTOS_MINIMOS) & ~hechos["grupo"].isin(GRUPOS_SIN_PERCENTIL)


# Calcula el percentil de una métrica dentro de cada grupo de posición
def percentil(tabla, columna):
    # Empezamos con todo vacío
    salida = pd.Series(np.nan, index=tabla.index)
    # Recorremos los grupos que sí llevan percentil
    for grupo in tabla.loc[~tabla["grupo"].isin(GRUPOS_SIN_PERCENTIL), "grupo"].unique():
        # Filas del grupo con dato en esta métrica
        en_grupo = (tabla["grupo"] == grupo) & tabla[columna].notna()
        # Filas de referencia del grupo
        ref = en_grupo & tabla["referencia"]
        # Valores de referencia ordenados
        orden = np.sort(tabla.loc[ref, columna].to_numpy())
        # Porcentaje de jugadores de referencia con un valor estrictamente menor
        salida[en_grupo] = np.searchsorted(orden, tabla.loc[en_grupo, columna].to_numpy(), side="left") / len(orden) * 100
    # Devolvemos la serie
    return salida


# Calculamos el percentil de cada métrica
for c in COLUMNAS_METRICA:
    # Columna con el percentil: pctl_ y el nombre de la métrica sin _90
    hechos["pctl_" + c.removesuffix("_90")] = percentil(hechos, c)

# Columnas de percentil
COLUMNAS_PCTL = ["pctl_" + c.removesuffix("_90") for c in COLUMNAS_METRICA]

# El percentil es fiable solo para los jugadores de referencia
hechos["percentil_fiable"] = hechos["referencia"]

# Jugadores de referencia de cada grupo de posición
n_ref = hechos[hechos["referencia"]].groupby("grupo").size()
# Cada jugador lleva el tamaño del grupo contra el que se calcula su percentil (vacío si no tiene percentil)
hechos["n_grupo_referencia"] = hechos["grupo"].map(n_ref).astype("Int64")
# Control: en todas las métricas el grupo de referencia es el mismo (nadie con 600 minutos tiene huecos)
for m in COLUMNAS_METRICA:
    # Jugadores de referencia con dato en esta métrica
    con_dato = hechos[hechos["referencia"] & hechos[m].notna()].groupby("grupo").size()
    # Tiene que coincidir con el total del grupo
    assert con_dato.equals(n_ref), f"grupo de referencia distinto en {m}"
# Umbral a partir del cual el panel avisa de que la muestra es pequeña
MUESTRA_REDUCIDA = 30
# Aviso para los grupos con 30 jugadores de referencia o menos
hechos["muestra_reducida"] = hechos["n_grupo_referencia"].le(MUESTRA_REDUCIDA).fillna(False).astype(bool)

# Columnas de la tabla de hechos
COLUMNAS_HECHOS = (["id_jugador", "tiene_eventos"] + COLUMNAS_METRICA
                   + COLUMNAS_PCTL + ["percentil_fiable", "n_grupo_referencia", "muestra_reducida"])
# Construimos la tabla de hechos
hechos_jugador = hechos[COLUMNAS_HECHOS].copy()
# Redondeamos las métricas a 4 decimales
hechos_jugador[COLUMNAS_METRICA] = hechos_jugador[COLUMNAS_METRICA].round(4)
# Redondeamos los percentiles a 2 decimales
hechos_jugador[COLUMNAS_PCTL] = hechos_jugador[COLUMNAS_PCTL].round(2)

## 8. Guardado

In [10]:
# Carpeta del modelo
os.makedirs("data/modelo", exist_ok=True)
# Guardamos cada tabla
for nombre, tabla in [("dim_equipo", dim_equipo), ("dim_posicion", dim_posicion),
                      ("dim_jugador", dim_jugador), ("hechos_jugador", hechos_jugador)]:
    # Escribimos el CSV sin índice
    tabla.to_csv(f"data/modelo/{nombre}.csv", index=False)
    # Informamos de filas y columnas
    print(f"{nombre}.csv: {tabla.shape[0]} filas, {tabla.shape[1]} columnas")

dim_equipo.csv: 20 filas, 7 columnas
dim_posicion.csv: 9 filas, 3 columnas
dim_jugador.csv: 561 filas, 10 columnas
hechos_jugador.csv: 561 filas, 47 columnas


## 9. Comprobaciones de calidad

In [11]:
# Releemos las tablas guardadas, que son las que verá Power BI
d_eq = pd.read_csv("data/modelo/dim_equipo.csv")
d_pos = pd.read_csv("data/modelo/dim_posicion.csv")
d_jug = pd.read_csv("data/modelo/dim_jugador.csv")
h = pd.read_csv("data/modelo/hechos_jugador.csv")

# 1. Jugadores en total y con percentil fiable
print("1. Jugadores en total:", len(d_jug), "· con percentil fiable:", int(h["percentil_fiable"].sum()))

# 2. Claves de jugador repetidas
print("2. id_jugador repetidos en dim_jugador:", int(d_jug["id_jugador"].duplicated().sum()),
      "· en hechos_jugador:", int(h["id_jugador"].duplicated().sum()))

# 3. Integridad referencial
print("3. Claves huérfanas:")
print("   hechos.id_jugador fuera de dim_jugador:", int((~h["id_jugador"].isin(d_jug["id_jugador"])).sum()))
print("   dim_jugador.id_equipo fuera de dim_equipo:", int((~d_jug["id_equipo"].isin(d_eq["id_equipo"])).sum()))
print("   dim_jugador.id_posicion fuera de dim_posicion:", int((~d_jug["id_posicion"].isin(d_pos["id_posicion"])).sum()))
print("   jugadores de dim_jugador sin fila de hechos:", int((~d_jug["id_jugador"].isin(h["id_jugador"])).sum()))

# 4. Jugadores por grupo de posición y cuántos pasan de 600 minutos
reparto = d_jug.merge(d_pos, on="id_posicion").assign(con_600=lambda t: t["minutos"] >= MINUTOS_MINIMOS)
# Tabla por grupo
por_grupo = reparto.groupby("grupo").agg(jugadores=("id_jugador", "size"), con_600_min=("con_600", "sum"))
print("\n4. Jugadores por grupo de posición")
print(por_grupo.sort_values("con_600_min", ascending=False).to_string())
# Grupos con percentil y muestra de referencia corta
cortos = por_grupo[(por_grupo["con_600_min"] <= 30) & ~por_grupo.index.isin(GRUPOS_SIN_PERCENTIL)]
print("   grupos con percentil y 30 o menos jugadores de referencia:", list(cortos.index) or "ninguno")

# 5. Valores ausentes por métrica
ausentes = h[[c for c in h.columns if c.endswith("_90") or c.startswith("pct_")]].isna().sum()
print("\n5. Métricas con valores ausentes (jugadores)")
print(ausentes[ausentes > 0].to_string() if (ausentes > 0).any() else "   ninguna")

1. Jugadores en total: 561 · con percentil fiable: 368
2. id_jugador repetidos en dim_jugador: 0 · en hechos_jugador: 0
3. Claves huérfanas:
   hechos.id_jugador fuera de dim_jugador: 0
   dim_jugador.id_equipo fuera de dim_equipo: 0
   dim_jugador.id_posicion fuera de dim_posicion: 0
   jugadores de dim_jugador sin fila de hechos: 0

4. Jugadores por grupo de posición
                    jugadores  con_600_min
grupo                                     
pivote_mediocentro        108           87
lateral                    94           73
central                    95           71
extremo                    86           59
delantero                  83           50
mediapunta                 39           28
sin_definir                28            5
sin_confirmar              28            0
   grupos con percentil y 30 o menos jugadores de referencia: ['mediapunta']

5. Métricas con valores ausentes (jugadores)
entradas_intentadas_90          5
entradas_exitosas_90            5
interce